# Phase 2b: Training LightGBM with SMOTE

This notebook tackles the class imbalance problem using the **Synthetic Minority Over-sampling Technique (SMOTE)**. The previous model, using `is_unbalance=True`, struggled to correctly classify underrepresented heading levels like `H1`, `H2`, and `H3`.

**Goal:** Improve the classification of minority classes by training a LightGBM model on a dataset balanced with SMOTE.

**Steps:**
1. Load the pre-processed dataset.
2. Split the data into training and testing sets.
3. **Apply SMOTE *only* to the training data** to create a balanced training set.
4. Train a new `LGBMClassifier` on the balanced data.
5. Evaluate the model on the original, untouched test set.

### 1. Import Necessary Libraries

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

# The key import for SMOTE
from imblearn.over_sampling import SMOTE

from sklearn.impute import SimpleImputer


print("Libraries imported successfully.")

### 2. Load and Prepare the Dataset

In [ ]:
DATASET_PATH = './AIH_model/AIH_preprocessor/final_training_dataset.csv'
df = pd.read_csv(DATASET_PATH)

X = df.drop('label', axis=1)
y = df['label']

# Define the label mapping for later visualization
label_map = {
    0: "BODY_TEXT", 1: "TITLE", 2: "H1",
    3: "H2", 4: "H3", 5: "H4"
}
label_names = [label_map[i] for i in sorted(y.unique())]

print("✅ Dataset loaded and prepared.")
print(f"X shape: {X.shape}, y shape: {y.shape}")

### 3. Split Data into Training and Testing Sets

This step is identical to before. We maintain a stratified split to ensure consistent class distributions in our train and test sets *before* resampling.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Data split into training and testing sets.")

### 4. Apply SMOTE to the Training Data

This is the most critical step. We apply SMOTE to oversample the minority classes in our **training set only**. This prevents data leakage, ensuring that our model is evaluated on real, unseen data.

We will print the class distribution before and after SMOTE to see its effect.

In [ ]:
print("Class distribution before SMOTE:")
print(y_train.value_counts().sort_index())
print(f"\nChecking for NaN values in X_train...")
nan_count = X_train.isnull().sum().sum()
print(f"Found {nan_count} NaN values in the training data.")

if nan_count > 0:
    print("Applying imputation to fill missing values...")
    # Initialize the imputer to replace NaNs with the median of each column
    imputer = SimpleImputer(strategy='median')
    
    # Fit the imputer on the training data and transform it
    # We use .fit_transform on the training data
    X_train_imputed = imputer.fit_transform(X_train)
    # IMPORTANT: We only use .transform on the test data to avoid data leakage
    X_test_imputed = imputer.transform(X_test)
    
    # Convert the imputed numpy arrays back to pandas DataFrames
    X_train = pd.DataFrame(X_train_imputed, columns=X.columns)
    X_test = pd.DataFrame(X_test_imputed, columns=X.columns)
    
    print("✅ Missing values handled.")
else:
    print("✅ No missing values found.")
print("\nApplying SMOTE to the imputed training data...")

# Find the number of samples in the smallest class
min_samples = y_train.value_counts().min()

# THE FIX: Set k_neighbors to be one less than the smallest class size.
# If min_samples is 2, k_neighbors must be 1.
k = min_samples - 1 if min_samples > 1 else 1
print(f"Smallest class has {min_samples} samples. Setting k_neighbors for SMOTE to {k}.")

# Initialize SMOTE
smote = SMOTE(random_state=42, k_neighbors=k)

# Fit and apply SMOTE
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\nClass distribution after SMOTE:")
print(y_train_resampled.value_counts().sort_index())

print(f"\nOriginal training set size: {len(X_train)}")
print(f"Resampled training set size: {len(X_train_resampled)}")

### 5. Train the LightGBM Model on Resampled Data

Now we train a new LightGBM model. Since the data is now balanced, we **do not** need to use the `is_unbalance=True` parameter. The model is trained on the new, larger, resampled training set.

In [ ]:
# Initialize a new model (without the is_unbalance parameter)
lgbm_smote = lgb.LGBMClassifier(random_state=42)

# Train the model on the resampled data
print("Training the LightGBM model on SMOTE-balanced data...")
lgbm_smote.fit(X_train_resampled, y_train_resampled)
print("✅ Model training complete.")

### 6. Evaluate Model Performance on the Original Test Set

We evaluate the new model's performance on the original, untouched `X_test` and `y_test`. This gives us an honest assessment of how well the model generalizes to real-world, imbalanced data.

In [ ]:
# Make predictions on the original test set
y_pred_smote = lgbm_smote.predict(X_test)

# --- Accuracy Score ---
accuracy = accuracy_score(y_test, y_pred_smote)
print(f"Overall Accuracy: {accuracy:.4f}\n")

# --- Classification Report ---
print("Classification Report (with SMOTE):")
print(classification_report(y_test, y_pred_smote, target_names=label_names))

# --- Confusion Matrix ---
print("Confusion Matrix (with SMOTE):")
cm = confusion_matrix(y_test, y_pred_smote)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix for LightGBM with SMOTE')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

### 7. Analysis and Next Steps

Compare this confusion matrix and classification report to the previous one. We are specifically looking for:

- **Improved Recall for Minority Classes**: Did the model correctly identify more `H1`, `H2`, `H3`, and `H4` samples?
- **Reduced Misclassifications**: Are there fewer instances of headings being misclassified as `BODY_TEXT`?

If performance has improved significantly, our next step will be to fine-tune the model's hyperparameters and then build the final prediction pipeline.